In [1]:
"""
01_train_winrate_model.py
===========================
Predicts whether a CLOSED deal is Won or Lost, and reports which
factors drive that outcome.

Data: clean_crm_data.csv (only closed deals are used, since is_won is
only meaningful once a deal has actually been decided)

Train/test split: 80/20, stratified on the target so the win/loss
ratio is preserved in both halves.

Output:
  saved_models/winrate_model.pkl
  saved_models/winrate_model_columns.pkl   (encoder + feature metadata)
"""

import pandas as pd
import numpy as np
import pickle
import os

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.preprocessing import OneHotEncoder

DATA_PATH = "clean_crm_data.csv"
OUT_DIR = "saved_models"
os.makedirs(OUT_DIR, exist_ok=True)

# ------------------------------------------------------------------
# Load + prepare
# ------------------------------------------------------------------
df = pd.read_csv(DATA_PATH)
closed = df[df["is_closed"] == 1].copy()
print(f"Closed deals: {len(closed):,}  |  Win rate: {closed['is_won'].mean():.1%}")

closed["engage_month"] = pd.to_datetime(closed["engage_date"]).dt.month
closed["engage_quarter"] = pd.to_datetime(closed["engage_date"]).dt.quarter

# Only use fields that are KNOWN AT DECISION TIME -- close_value and
# deal_stage are outcomes of the deal being won/lost, so they must NOT
# be used as inputs (that would be data leakage and give a fake-perfect
# score that falls apart on new deals).
feature_cols_num = ["deal_value_proposed", "employees",
                     "sales_cycle_days", "engage_month", "engage_quarter"]
feature_cols_cat = ["sector", "product", "regional_office",
                     "office_location", "sales_agent"]

X_num = closed[feature_cols_num].fillna(closed[feature_cols_num].median())
enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
X_cat = enc.fit_transform(closed[feature_cols_cat])
cat_feature_names = enc.get_feature_names_out(feature_cols_cat)

X = np.hstack([X_num.values, X_cat])
feature_names = feature_cols_num + list(cat_feature_names)
y = closed["is_won"].values

# ------------------------------------------------------------------
# Train / test split (80/20, stratified)
# ------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_train):,}   Test: {len(X_test):,}")

# ------------------------------------------------------------------
# Model + 5-fold cross-validation on the TRAIN split only
# (gives an honest sense of stability before ever touching test data)
# ------------------------------------------------------------------
clf = RandomForestClassifier(
    n_estimators=400, max_depth=12, min_samples_leaf=10,
    class_weight="balanced_subsample", random_state=42, n_jobs=-1
)

cv_scores = cross_val_score(clf, X_train, y_train, cv=5, scoring="roc_auc")
print(f"\n5-fold CV ROC-AUC on training data: {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}")

# ------------------------------------------------------------------
# Fit on full train split, evaluate once on the held-out test split
# ------------------------------------------------------------------
clf.fit(X_train, y_train)
y_prob = clf.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

auc = roc_auc_score(y_test, y_prob)
print(f"\n=== Held-out TEST performance ===")
print(f"ROC-AUC: {auc:.3f}  (0.5 = random guessing, 1.0 = perfect)")
print(classification_report(y_test, y_pred, target_names=["Lost", "Won"]))
print("Confusion matrix (rows=actual, cols=predicted):")
print(confusion_matrix(y_test, y_pred))

# ------------------------------------------------------------------
# Feature importance (rolled up from one-hot back to parent column)
# ------------------------------------------------------------------
importances = pd.Series(clf.feature_importances_, index=feature_names)
rollup = {}
for name, val in importances.items():
    parent = name
    for cat_col in feature_cols_cat:
        if name.startswith(cat_col + "_"):
            parent = cat_col
            break
    rollup[parent] = rollup.get(parent, 0) + val
rollup = pd.Series(rollup).sort_values(ascending=False)
print("\n=== What most influences win/loss ===")
print(rollup.round(3))

# ------------------------------------------------------------------
# Save
# ------------------------------------------------------------------
with open(os.path.join(OUT_DIR, "winrate_model.pkl"), "wb") as f:
    pickle.dump(clf, f)
with open(os.path.join(OUT_DIR, "winrate_model_columns.pkl"), "wb") as f:
    pickle.dump({
        "feature_cols_num": feature_cols_num,
        "feature_cols_cat": feature_cols_cat,
        "encoder": enc,
        "feature_names": feature_names,
    }, f)
print(f"\nSaved -> {OUT_DIR}/winrate_model.pkl, {OUT_DIR}/winrate_model_columns.pkl")

Closed deals: 22,630  |  Win rate: 44.5%
Train: 18,104   Test: 4,526

5-fold CV ROC-AUC on training data: 0.644 +/- 0.007

=== Held-out TEST performance ===
ROC-AUC: 0.637  (0.5 = random guessing, 1.0 = perfect)
              precision    recall  f1-score   support

        Lost       0.68      0.47      0.56      2514
         Won       0.52      0.73      0.61      2012

    accuracy                           0.59      4526
   macro avg       0.60      0.60      0.58      4526
weighted avg       0.61      0.59      0.58      4526

Confusion matrix (rows=actual, cols=predicted):
[[1184 1330]
 [ 548 1464]]

=== What most influences win/loss ===
sales_cycle_days       0.262
deal_value_proposed    0.233
sector                 0.185
employees              0.080
sales_agent            0.078
product                0.043
office_location        0.038
engage_month           0.036
regional_office        0.026
engage_quarter         0.018
dtype: float64

Saved -> saved_models/winrate_model.pkl, 

In [3]:
"""
02_train_revenue_forecast_model.py
=====================================
Forecasts next-quarter (3 month) company-wide revenue from the monthly
won-deal time series.

Train/test split for time series is NOT random -- you must split
chronologically, or the model "sees the future" while training. Here
we hold out the LAST 6 real months as a test set, fit on everything
before that, forecast those 6 months, and score the forecast against
what actually happened.

Output:
  saved_models/revenue_forecast_model.pkl   (final model, fit on ALL data)
  saved_models/monthly_revenue_series.pkl
"""

import pandas as pd
import numpy as np
import pickle
import os

from statsmodels.tsa.holtwinters import ExponentialSmoothing

DATA_PATH = "clean_crm_data.csv"
OUT_DIR = "saved_models"
os.makedirs(OUT_DIR, exist_ok=True)

# ------------------------------------------------------------------
# Build monthly revenue series (won deals only)
# ------------------------------------------------------------------
df = pd.read_csv(DATA_PATH)
won = df[df["is_won"] == 1].copy()
won["close_date"] = pd.to_datetime(won["close_date"])

monthly = (won.set_index("close_date")
              .resample("MS")["close_value"]
              .sum())
monthly = monthly.asfreq("MS", fill_value=0)
print("=== Monthly revenue series ===")
print(f"{len(monthly)} months, {monthly.index.min().date()} -> {monthly.index.max().date()}")

# Drop the most recent month if it looks incomplete (partial pipeline tail)
# -- same logic as before: a live extract's final month is naturally low
# because deals engaged recently haven't finished closing yet.
if monthly.iloc[-1] < monthly.iloc[-6:-1].mean() * 0.5:
    print(f"Dropping likely-incomplete trailing month: {monthly.index[-1].date()}")
    monthly = monthly.iloc[:-1]

# ------------------------------------------------------------------
# Chronological train/test split: hold out the last 6 months
# ------------------------------------------------------------------
TEST_MONTHS = 6
train_series = monthly.iloc[:-TEST_MONTHS]
test_series = monthly.iloc[-TEST_MONTHS:]
print(f"\nTrain: {len(train_series)} months  |  Test (held out): {len(test_series)} months")

# ------------------------------------------------------------------
# Fit on train only, forecast the held-out months, score honestly
# ------------------------------------------------------------------
def fit_hw(series, seasonal_periods=12):
    try:
        model = ExponentialSmoothing(
            series, trend="add", damped_trend=True, seasonal="add",
            seasonal_periods=seasonal_periods, initialization_method="estimated"
        ).fit(optimized=True)
        return model
    except Exception as e:
        print(f"Seasonal fit failed ({e}), falling back to trend-only")
        return ExponentialSmoothing(
            series, trend="add", damped_trend=True, seasonal=None,
            initialization_method="estimated"
        ).fit(optimized=True)

bt_model = fit_hw(train_series)
bt_forecast = bt_model.forecast(TEST_MONTHS).clip(lower=0)

mae = float(np.mean(np.abs(test_series.values - bt_forecast.values)))
mape = float(np.mean(np.abs((test_series.values - bt_forecast.values) /
                             np.where(test_series.values == 0, 1, test_series.values))) * 100)

print("\n=== Backtest on held-out months ===")
print(pd.DataFrame({
    "actual": test_series.values.round(0),
    "predicted": bt_forecast.values.round(0),
    "abs_error": np.abs(test_series.values - bt_forecast.values).round(0),
}, index=test_series.index))
print(f"\nMAE:  {mae:,.0f}")
print(f"MAPE: {mape:.1f}%  (mean absolute percentage error)")

# ------------------------------------------------------------------
# Refit on ALL available data for the actual production forecast
# ------------------------------------------------------------------
final_model = fit_hw(monthly)
next_3_months = final_model.forecast(3).clip(lower=0)
print("\n=== Next 3 months forecast (production model, fit on all data) ===")
print(next_3_months.round(0))

# 80%/95% confidence intervals via residual simulation
n_sims = 2000
sims = final_model.simulate(3, repetitions=n_sims, error="add", random_state=42)
lower_80, upper_80 = sims.quantile(0.10, axis=1), sims.quantile(0.90, axis=1)
lower_95, upper_95 = sims.quantile(0.025, axis=1), sims.quantile(0.975, axis=1)

forecast_df = pd.DataFrame({
    "month": next_3_months.index,
    "forecast_revenue": next_3_months.values.round(0),
    "lower_80": lower_80.clip(lower=0).values.round(0),
    "upper_80": upper_80.values.round(0),
    "lower_95": lower_95.clip(lower=0).values.round(0),
    "upper_95": upper_95.values.round(0),
})
forecast_df.to_csv(os.path.join(OUT_DIR, "revenue_forecast_next_3_months.csv"), index=False)
print(f"\nSaved -> {OUT_DIR}/revenue_forecast_next_3_months.csv")

# ------------------------------------------------------------------
# Save
# ------------------------------------------------------------------
with open(os.path.join(OUT_DIR, "revenue_forecast_model.pkl"), "wb") as f:
    pickle.dump(final_model, f)
with open(os.path.join(OUT_DIR, "monthly_revenue_series.pkl"), "wb") as f:
    pickle.dump(monthly, f)
print(f"Saved -> {OUT_DIR}/revenue_forecast_model.pkl, {OUT_DIR}/monthly_revenue_series.pkl")
print(f"\nBacktest MAPE was {mape:.1f}% -- treat that as the expected error on new forecasts.")

=== Monthly revenue series ===
51 months, 2022-02-01 -> 2026-04-01
Dropping likely-incomplete trailing month: 2026-04-01

Train: 44 months  |  Test (held out): 6 months

=== Backtest on held-out months ===
               actual  predicted  abs_error
close_date                                 
2025-10-01  1218571.0  1221617.0     3046.0
2025-11-01  1225032.0  1146928.0    78104.0
2025-12-01  1211301.0  1154712.0    56589.0
2026-01-01  1056928.0  1201083.0   144155.0
2026-02-01   956711.0  1080375.0   123664.0
2026-03-01   554319.0  1272316.0   717997.0

MAE:  187,259
MAPE: 27.9%  (mean absolute percentage error)

=== Next 3 months forecast (production model, fit on all data) ===
2026-04-01    1066928.0
2026-05-01    1164756.0
2026-06-01     962212.0
Freq: MS, dtype: float64

Saved -> saved_models/revenue_forecast_next_3_months.csv
Saved -> saved_models/revenue_forecast_model.pkl, saved_models/monthly_revenue_series.pkl

Backtest MAPE was 27.9% -- treat that as the expected error on new 

In [6]:
"""
03_train_employee_quarterly_model.py
=======================================
Predicts a sales rep's NEXT quarter revenue from their trailing
performance stats this quarter (deals worked, win rate, avg deal size,
avg cycle length, revenue this quarter).

Uses "sales_agent" as the rep identifier (clean_crm_data.csv has no
separate rep_id column -- sales_agent names are unique per rep here).

Train/test split: random 80/20 split of rep-quarter rows. This is fine
here (unlike the revenue forecast) because each row is one rep's one
quarter, and rows are independent enough across reps/quarters -- but we
still avoid leaking a rep's OWN future quarter into their own training
row (the target is always a full quarter ahead of the features).

Output:
  saved_models/employee_quarterly_model.pkl
  saved_models/employee_quarterly_features.pkl
  saved_models/rep_quarter_panel.csv
"""

import pandas as pd
import numpy as np
import pickle
import os

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score

DATA_PATH = "clean_crm_data.csv"
OUT_DIR = "saved_models"
os.makedirs(OUT_DIR, exist_ok=True)

REP_COL = "sales_agent"

# ------------------------------------------------------------------
# Build a rep x quarter panel
# ------------------------------------------------------------------
df = pd.read_csv(DATA_PATH)
df["engage_date"] = pd.to_datetime(df["engage_date"])
df["close_date"] = pd.to_datetime(df["close_date"], errors="coerce")
df["engage_quarter"] = df["engage_date"].dt.to_period("Q")

won = df[df["is_won"] == 1].copy()
won["close_quarter"] = won["close_date"].dt.to_period("Q")

rep_quarter_revenue = (won.groupby([REP_COL, "close_quarter"])["revenue"]
                       .sum().rename("revenue"))

rep_quarter_stats = df.groupby([REP_COL, "engage_quarter"]).agg(
    deals_worked=("opportunity_id", "count"),
    win_rate=("is_won", "mean"),
    avg_deal_size=("deal_value_proposed", "mean"),
    avg_cycle_days=("sales_cycle_days", "mean"),
).rename_axis(index=[REP_COL, "quarter"])

panel = rep_quarter_stats.join(
    rep_quarter_revenue.rename_axis(index=[REP_COL, "quarter"]), how="left"
).fillna(0.0).reset_index()
panel = panel.sort_values([REP_COL, "quarter"])

# Target = NEXT quarter's revenue for that same rep
panel["next_quarter_revenue"] = panel.groupby(REP_COL)["revenue"].shift(-1)
train_panel = panel.dropna(subset=["next_quarter_revenue"]).copy()

print(f"Rep-quarter rows available for training: {len(train_panel):,} "
      f"(across {train_panel[REP_COL].nunique()} reps)")

feature_cols = ["deals_worked", "win_rate", "avg_deal_size", "avg_cycle_days", "revenue"]
X = train_panel[feature_cols]
y = train_panel["next_quarter_revenue"]

# ------------------------------------------------------------------
# Train / test split (80/20)
# ------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {len(X_train):,}   Test: {len(X_test):,}")

# ------------------------------------------------------------------
# Model + cross-validation on train only
# ------------------------------------------------------------------
model = RandomForestRegressor(
    n_estimators=400, max_depth=8, min_samples_leaf=5,
    random_state=42, n_jobs=-1
)
cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring="r2")
print(f"\n5-fold CV R2 on training data: {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}")

# ------------------------------------------------------------------
# Fit + evaluate on held-out test split
# ------------------------------------------------------------------
model.fit(X_train, y_train)
pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)
# MAPE only over rows where actual revenue isn't 0, to avoid divide-by-zero blowups
nonzero = y_test != 0
mape = mean_absolute_percentage_error(y_test[nonzero], pred[nonzero]) * 100

print(f"\n=== Held-out TEST performance ===")
print(f"MAE:  {mae:,.2f}")
print(f"MAPE: {mape:.1f}%")
print(f"R2:   {r2:.3f}")

print("\n=== Feature importance ===")
for f, imp in sorted(zip(feature_cols, model.feature_importances_), key=lambda x: -x[1]):
    print(f"  {f}: {imp:.3f}")

# ------------------------------------------------------------------
# Save
# ------------------------------------------------------------------
with open(os.path.join(OUT_DIR, "employee_quarterly_model.pkl"), "wb") as f:
    pickle.dump(model, f)
with open(os.path.join(OUT_DIR, "employee_quarterly_features.pkl"), "wb") as f:
    pickle.dump(feature_cols, f)
panel.to_csv(os.path.join(OUT_DIR, "rep_quarter_panel.csv"), index=False)
print(f"\nSaved -> {OUT_DIR}/employee_quarterly_model.pkl, "
      f"{OUT_DIR}/employee_quarterly_features.pkl, {OUT_DIR}/rep_quarter_panel.csv")

Rep-quarter rows available for training: 525 (across 35 reps)
Train: 420   Test: 105

5-fold CV R2 on training data: 0.565 +/- 0.055

=== Held-out TEST performance ===
MAE:  693,743.02
MAPE: 14.2%
R2:   0.539

=== Feature importance ===
  deals_worked: 0.554
  win_rate: 0.304
  avg_deal_size: 0.057
  avg_cycle_days: 0.046
  revenue: 0.039

Saved -> saved_models/employee_quarterly_model.pkl, saved_models/employee_quarterly_features.pkl, saved_models/rep_quarter_panel.csv


In [7]:
"""
04_train_employee_yearly_model.py
====================================
Predicts a sales rep's revenue trend into future years (used to project
1-3 years ahead per rep), from their tenure, deal volume, win rate, and
current-year revenue.

Uses "sales_agent" as the rep identifier.

Train/test split: 80/20 random split of rep-year rows, with the same
"predict next year from this year" structure as the quarterly model.

Output:
  saved_models/employee_yearly_model.pkl
  saved_models/employee_yearly_features.pkl
  saved_models/rep_year_panel.csv
"""

import pandas as pd
import numpy as np
import pickle
import os

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score

DATA_PATH = "clean_crm_data.csv"
OUT_DIR = "saved_models"
os.makedirs(OUT_DIR, exist_ok=True)

REP_COL = "sales_agent"

# ------------------------------------------------------------------
# Build a rep x year panel
# ------------------------------------------------------------------
df = pd.read_csv(DATA_PATH)
df["engage_date"] = pd.to_datetime(df["engage_date"])
df["close_date"] = pd.to_datetime(df["close_date"], errors="coerce")

won = df[df["is_won"] == 1].copy()
won["close_year"] = won["close_date"].dt.year

rep_year_revenue = won.groupby([REP_COL, "close_year"])["revenue"].sum().reset_index()

# Tenure: first year each rep appears in the data
rep_first_year = df.groupby(REP_COL)["engage_date"].min().dt.year.rename("first_active_year")
rep_year_revenue = rep_year_revenue.merge(rep_first_year, on=REP_COL, how="left")
rep_year_revenue["years_active"] = rep_year_revenue["close_year"] - rep_year_revenue["first_active_year"]

rep_year_deals = df.groupby([REP_COL, df["engage_date"].dt.year]).agg(
    deals_worked=("opportunity_id", "count"),
    win_rate=("is_won", "mean"),
).rename_axis(index=[REP_COL, "close_year"]).reset_index()

panel = rep_year_revenue.merge(rep_year_deals, on=[REP_COL, "close_year"], how="left").fillna(0)
panel = panel.sort_values([REP_COL, "close_year"])

# Target = NEXT year's revenue for that same rep
panel["next_year_revenue"] = panel.groupby(REP_COL)["revenue"].shift(-1)
train_panel = panel.dropna(subset=["next_year_revenue"]).copy()

print(f"Rep-year rows available for training: {len(train_panel):,} "
      f"(across {train_panel[REP_COL].nunique()} reps)")

feature_cols = ["revenue", "years_active", "deals_worked", "win_rate"]
X = train_panel[feature_cols]
y = train_panel["next_year_revenue"]

# ------------------------------------------------------------------
# Train / test split (80/20)
# ------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {len(X_train):,}   Test: {len(X_test):,}")

if len(X_train) < 30:
    print("\nNOTE: very few rep-year rows are available (each rep only has a "
          "handful of full years in this dataset). Metrics below may be noisy "
          "-- treat this model as directional, not precise, until more years "
          "of history accumulate.")

# ------------------------------------------------------------------
# Model + cross-validation
# ------------------------------------------------------------------
model = GradientBoostingRegressor(
    n_estimators=300, max_depth=3, learning_rate=0.05, random_state=42
)
cv_folds = min(5, max(2, len(X_train) // 10))
cv_scores = cross_val_score(model, X_train, y_train, cv=cv_folds, scoring="r2")
print(f"\n{cv_folds}-fold CV R2 on training data: {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}")

# ------------------------------------------------------------------
# Fit + evaluate on held-out test split
# ------------------------------------------------------------------
model.fit(X_train, y_train)
pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)
nonzero = y_test != 0
mape = mean_absolute_percentage_error(y_test[nonzero], pred[nonzero]) * 100 if nonzero.any() else float("nan")

print(f"\n=== Held-out TEST performance ===")
print(f"MAE:  {mae:,.2f}")
print(f"MAPE: {mape:.1f}%")
print(f"R2:   {r2:.3f}")

print("\n=== Feature importance ===")
for f, imp in sorted(zip(feature_cols, model.feature_importances_), key=lambda x: -x[1]):
    print(f"  {f}: {imp:.3f}")

# ------------------------------------------------------------------
# Save
# ------------------------------------------------------------------
with open(os.path.join(OUT_DIR, "employee_yearly_model.pkl"), "wb") as f:
    pickle.dump(model, f)
with open(os.path.join(OUT_DIR, "employee_yearly_features.pkl"), "wb") as f:
    pickle.dump(feature_cols, f)
panel.to_csv(os.path.join(OUT_DIR, "rep_year_panel.csv"), index=False)
print(f"\nSaved -> {OUT_DIR}/employee_yearly_model.pkl, "
      f"{OUT_DIR}/employee_yearly_features.pkl, {OUT_DIR}/rep_year_panel.csv")

Rep-year rows available for training: 140 (across 35 reps)
Train: 112   Test: 28

5-fold CV R2 on training data: 0.915 +/- 0.028

=== Held-out TEST performance ===
MAE:  1,999,890.94
MAPE: 14.8%
R2:   0.912

=== Feature importance ===
  years_active: 0.919
  deals_worked: 0.032
  win_rate: 0.027
  revenue: 0.022

Saved -> saved_models/employee_yearly_model.pkl, saved_models/employee_yearly_features.pkl, saved_models/rep_year_panel.csv


In [8]:
"""
05_train_lag_feature_revenue_model.py
========================================
A revenue prediction model built on ENGINEERED LAG FEATURES instead of
Holt-Winters. This is the API-friendly version: instead of holding the
whole time series internally, it takes a row of numbers
(lag_1, lag_2, lag_3, lag_6, lag_12, rolling_mean_3, rolling_std_3,
month, quarter) and predicts next month's revenue -- exactly the shape
you'd send from a frontend form or a backend request body.

------------------------------------------------------------------
IMPORTANT / HONEST NOTE ON DATA SIZE
------------------------------------------------------------------
Your monthly revenue series only has ~50 months of data, and the last
few months are an artifact of the synthetic generator (engage_dates only
go up to Dec 2025, so revenue closing in 2026 thins out for reasons that
have nothing to do with a real business trend -- it's not a real decline,
it's the data running out). This script auto-detects and trims that fake
tail before training. After trimming and building lag features (which
eat the first 12 months as warm-up), you're left with roughly ~30-35
usable training rows. That is a SMALL dataset for machine learning.

I'm not going to tell you this is "perfectly accurate" -- with this
little data, no model is. What this script gives you is: an honest
backtest score so you know exactly how much to trust it, and a solid,
leak-free feature pipeline that will get MUCH better the moment you
feed it real, ongoing monthly data (12+ more months = dramatically
more reliable).

Output:
  saved_models/lag_revenue_model.pkl
  saved_models/lag_revenue_features.pkl
  saved_models/lag_revenue_backtest.csv
"""

import pandas as pd
import numpy as np
import pickle
import os

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score

DATA_PATH = "clean_crm_data.csv"
OUT_DIR = "saved_models"
os.makedirs(OUT_DIR, exist_ok=True)

# ------------------------------------------------------------------
# Build monthly revenue series
# ------------------------------------------------------------------
df = pd.read_csv(DATA_PATH)
won = df[df["is_won"] == 1].copy()
won["close_date"] = pd.to_datetime(won["close_date"])

monthly = (won.set_index("close_date")
              .resample("MS")["close_value"]
              .sum())
monthly = monthly.asfreq("MS", fill_value=0)
print(f"Raw monthly series: {len(monthly)} months "
      f"({monthly.index.min().date()} -> {monthly.index.max().date()})")

# ------------------------------------------------------------------
# Auto-trim the fake trailing tail: walking backward from the end,
# drop any month that's less than 55% of the mean of the 6 months
# before it. This is a pipeline-thinning artifact, not a real trend.
# ------------------------------------------------------------------
def trim_incomplete_tail(series, threshold=0.55, lookback=6):
    s = series.copy()
    while len(s) > lookback + 1:
        recent_mean = s.iloc[-(lookback + 1):-1].mean()
        if s.iloc[-1] < threshold * recent_mean:
            print(f"  Dropping incomplete month: {s.index[-1].date()} "
                  f"({s.iloc[-1]:,.0f} vs recent avg {recent_mean:,.0f})")
            s = s.iloc[:-1]
        else:
            break
    return s

monthly = trim_incomplete_tail(monthly)
print(f"After trimming: {len(monthly)} months "
      f"({monthly.index.min().date()} -> {monthly.index.max().date()})")

# ------------------------------------------------------------------
# Feature engineering: lag_1, lag_2, lag_3, lag_6, lag_12,
# rolling_mean_3, rolling_std_3, month, quarter
# ------------------------------------------------------------------
def build_features(series):
    frame = pd.DataFrame({"revenue": series})
    frame["lag_1"] = frame["revenue"].shift(1)
    frame["lag_2"] = frame["revenue"].shift(2)
    frame["lag_3"] = frame["revenue"].shift(3)
    frame["lag_6"] = frame["revenue"].shift(6)
    frame["lag_12"] = frame["revenue"].shift(12)
    frame["rolling_mean_3"] = frame["revenue"].shift(1).rolling(3).mean()
    frame["rolling_std_3"] = frame["revenue"].shift(1).rolling(3).std()
    frame["month"] = frame.index.month
    frame["quarter"] = frame.index.quarter
    return frame

feat = build_features(monthly)
feature_cols = ["lag_1", "lag_2", "lag_3", "lag_6", "lag_12",
                 "rolling_mean_3", "rolling_std_3", "month", "quarter"]

feat_clean = feat.dropna(subset=feature_cols)  # drops the first 12 warm-up months
print(f"\nUsable training rows after lag warm-up: {len(feat_clean)}")

if len(feat_clean) < 15:
    print("WARNING: fewer than 15 usable rows. This model will be highly "
          "unstable. Strongly recommend collecting more months of real "
          "data before relying on this for decisions.")

X = feat_clean[feature_cols]
y = feat_clean["revenue"]

# ------------------------------------------------------------------
# Chronological train/test split (never random for time series):
# last 20% of rows (by time) held out as test
# ------------------------------------------------------------------
n_test = max(3, int(len(X) * 0.2))
X_train, X_test = X.iloc[:-n_test], X.iloc[-n_test:]
y_train, y_test = y.iloc[:-n_test], y.iloc[-n_test:]
print(f"Train: {len(X_train)} rows   Test: {len(X_test)} rows (most recent, held out)")

# ------------------------------------------------------------------
# Walk-forward backtest: retrain at each step using only data before
# that point, predict one month ahead -- this is the honest way to
# score a time-series model (no peeking at future rows during training)
# ------------------------------------------------------------------
wf_preds, wf_actuals, wf_dates = [], [], []
for i in range(len(X) - n_test, len(X)):
    X_tr, y_tr = X.iloc[:i], y.iloc[:i]
    if len(X_tr) < 5:
        continue
    m = GradientBoostingRegressor(n_estimators=150, max_depth=2,
                                   learning_rate=0.05, random_state=42)
    m.fit(X_tr, y_tr)
    pred = m.predict(X.iloc[[i]])[0]
    wf_preds.append(pred)
    wf_actuals.append(y.iloc[i])
    wf_dates.append(X.index[i])

wf_preds, wf_actuals = np.array(wf_preds), np.array(wf_actuals)
mae = mean_absolute_error(wf_actuals, wf_preds)
mape = np.mean(np.abs((wf_actuals - wf_preds) / np.where(wf_actuals == 0, 1, wf_actuals))) * 100
r2 = r2_score(wf_actuals, wf_preds) if len(wf_actuals) > 1 else float("nan")

backtest_df = pd.DataFrame({
    "month": wf_dates, "actual": wf_actuals.round(0), "predicted": wf_preds.round(0),
    "abs_error": np.abs(wf_actuals - wf_preds).round(0),
})
print("\n=== Walk-forward backtest (honest, no leakage) ===")
print(backtest_df.to_string(index=False))
print(f"\nMAE:  {mae:,.0f}")
print(f"MAPE: {mape:.1f}%")
print(f"R2:   {r2:.3f}")
backtest_df.to_csv(os.path.join(OUT_DIR, "lag_revenue_backtest.csv"), index=False)

# ------------------------------------------------------------------
# Final production model: fit on ALL available rows
# ------------------------------------------------------------------
final_model = GradientBoostingRegressor(n_estimators=150, max_depth=2,
                                         learning_rate=0.05, random_state=42)
final_model.fit(X, y)

print("\n=== Feature importance (final model) ===")
for f, imp in sorted(zip(feature_cols, final_model.feature_importances_), key=lambda x: -x[1]):
    print(f"  {f}: {imp:.3f}")

with open(os.path.join(OUT_DIR, "lag_revenue_model.pkl"), "wb") as f:
    pickle.dump(final_model, f)
with open(os.path.join(OUT_DIR, "lag_revenue_features.pkl"), "wb") as f:
    pickle.dump(feature_cols, f)
print(f"\nSaved -> {OUT_DIR}/lag_revenue_model.pkl, {OUT_DIR}/lag_revenue_features.pkl")

# ------------------------------------------------------------------
# Example: this is the exact input shape your frontend/backend needs
# to send to get a prediction (see 05b_predict_example.py for a
# ready-to-use function).
# ------------------------------------------------------------------
last_row = feat.iloc[[-1]][feature_cols]
example_input = last_row.to_dict(orient="records")[0]
print("\n=== Example input for next-month prediction (from your real latest data) ===")
print(example_input)
example_pred = final_model.predict(last_row)[0]
print(f"Predicted next month revenue: {example_pred:,.0f}")

Raw monthly series: 51 months (2022-02-01 -> 2026-04-01)
  Dropping incomplete month: 2026-04-01 (40,608 vs recent avg 1,037,144)
  Dropping incomplete month: 2026-03-01 (554,319 vs recent avg 1,152,653)
After trimming: 49 months (2022-02-01 -> 2026-02-01)

Usable training rows after lag warm-up: 37
Train: 30 rows   Test: 7 rows (most recent, held out)

=== Walk-forward backtest (honest, no leakage) ===
     month    actual  predicted  abs_error
2025-08-01 1175308.0  1096457.0    78851.0
2025-09-01 1247372.0  1154143.0    93230.0
2025-10-01 1218571.0  1177690.0    40881.0
2025-11-01 1225032.0  1109022.0   116010.0
2025-12-01 1211301.0  1297570.0    86270.0
2026-01-01 1056928.0  1150545.0    93618.0
2026-02-01  956711.0  1062400.0   105689.0

MAE:  87,793
MAPE: 7.7%
R2:   0.180

=== Feature importance (final model) ===
  lag_12: 0.607
  lag_1: 0.098
  rolling_std_3: 0.070
  lag_3: 0.069
  month: 0.056
  lag_2: 0.026
  quarter: 0.026
  rolling_mean_3: 0.025
  lag_6: 0.022

Saved -> saved

In [9]:
"""
05b_predict_example.py
=========================
Drop this logic straight into your backend. It shows exactly how to
load the saved lag-feature model and turn a JSON-style input (the
numbers your frontend collects) into a revenue prediction.

This is the function to wrap in a FastAPI/Flask endpoint.
"""

import pickle
import pandas as pd

OUT_DIR = "saved_models"

with open(f"{OUT_DIR}/lag_revenue_model.pkl", "rb") as f:
    model = pickle.load(f)
with open(f"{OUT_DIR}/lag_revenue_features.pkl", "rb") as f:
    feature_cols = pickle.load(f)


def predict_next_month_revenue(lag_1, lag_2, lag_3, lag_6, lag_12,
                                rolling_mean_3, rolling_std_3,
                                month, quarter):
    """
    All 9 arguments are required, in the units your revenue column is in.

      lag_1           = last month's revenue
      lag_2           = revenue 2 months ago
      lag_3           = revenue 3 months ago
      lag_6           = revenue 6 months ago
      lag_12          = revenue 12 months ago (same month last year)
      rolling_mean_3  = average revenue over the last 3 months
      rolling_std_3   = standard deviation of revenue over the last 3 months
      month           = calendar month you're predicting FOR (1-12)
      quarter         = calendar quarter you're predicting FOR (1-4)

    Returns: predicted revenue (float) for the next month.
    """
    row = pd.DataFrame([{
        "lag_1": lag_1, "lag_2": lag_2, "lag_3": lag_3,
        "lag_6": lag_6, "lag_12": lag_12,
        "rolling_mean_3": rolling_mean_3, "rolling_std_3": rolling_std_3,
        "month": month, "quarter": quarter,
    }])[feature_cols]  # enforce exact training column order
    return float(model.predict(row)[0])


# ------------------------------------------------------------------
# Example call -- this is the same shape a frontend form would submit
# ------------------------------------------------------------------
if __name__ == "__main__":
    example_input = {
        "lag_1": 1056927.72,
        "lag_2": 1211300.64,
        "lag_3": 1225032.40,
        "lag_6": 1175308.42,
        "lag_12": 959930.92,
        "rolling_mean_3": 1164420.25,
        "rolling_std_3": 93344.12,
        "month": 2,
        "quarter": 1,
    }
    prediction = predict_next_month_revenue(**example_input)
    print("Input:", example_input)
    print(f"Predicted next-month revenue: {prediction:,.0f}")

    # ------------------------------------------------------------------
    # Minimal FastAPI wrapper you can copy into your backend as-is:
    # ------------------------------------------------------------------
    """
    from fastapi import FastAPI
    from pydantic import BaseModel

    app = FastAPI()

    class RevenueInput(BaseModel):
        lag_1: float
        lag_2: float
        lag_3: float
        lag_6: float
        lag_12: float
        rolling_mean_3: float
        rolling_std_3: float
        month: int
        quarter: int

    @app.post("/predict/revenue")
    def predict_revenue(data: RevenueInput):
        prediction = predict_next_month_revenue(**data.dict())
        return {"predicted_revenue": prediction}
    """

Input: {'lag_1': 1056927.72, 'lag_2': 1211300.64, 'lag_3': 1225032.4, 'lag_6': 1175308.42, 'lag_12': 959930.92, 'rolling_mean_3': 1164420.25, 'rolling_std_3': 93344.12, 'month': 2, 'quarter': 1}
Predicted next-month revenue: 971,393
